In [1]:
!pip install pandas
!pip install chromadb
!pip install sentence-transformers
!pip install torch
!pip install gradio
!pip install kagglehub[pandas-datasets]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [3]:
from google.colab import userdata

# Load Kaggle API credentials from Colab secrets
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle API credentials loaded.")

Kaggle API credentials loaded.


In [4]:
import os
import pandas as pd
import kagglehub

from kagglehub import KaggleDatasetAdapter

# Kaggle dataset + file inside the dataset (must include extension)

dataset_ref = "sirishk99/cmrti-placement-questions-dataset"
file_path = "undergrad-cs-questions-data.parquet"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    dataset_ref,
    file_path,
)


columns_to_drop = [
    col for col in ["relative_path", "source_type", "file_size", "file_size_kb"]
    if col in df.columns
]

df = df.drop(columns=columns_to_drop)
print("Loaded shape:", df.shape)
print(df.head())

data = df.to_dict(orient="records")
print("Sample record:", data[0])


/tmp/ipykernel_732/2683213627.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 325k/325k [00:00<00:00, 1.25MB/s]

Loaded shape: (50, 3)
                                            filename  \
0                             PTQB 2025-26_AInDS.csv   
1  CSE_ technical and programming questions place...   
2                             ISE PTQB 2025-2026.csv   
3                         ISE PTQB 2024-2025 (1).csv   
4                             PTQB 2024-25_AInDS.csv   

                                                text  text_length  
0              company name                      ...        76811  
1              company name                      ...       165179  
2       company name                             ...        19231  
3                        company name            ...       261225  
4             company name                       ...        37907  
Sample record: {'filename': 'PTQB 2025-26_AInDS.csv', 'text': '            company name                                                                                                                                                  

In [7]:
#Chunk the data
def chunk_data(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunked_data = []
for record in data:
    text = record['text']
    chunks = chunk_data(text)
    for i, chunk in enumerate(chunks):
        chunked_record = record.copy()
        chunked_record['text'] = chunk
        chunked_record['chunk_index'] = i
        chunked_data.append(chunked_record)
print("Total chunks created:", len(chunked_data))
print("Sample chunked record:", chunked_data[0])

Total chunks created: 10186
Sample chunked record: {'filename': 'PTQB 2025-26_AInDS.csv', 'text': '            company name                                                                                                                                                                                ', 'text_length': 76811, 'chunk_index': 0}


### Initialize chroma vector database client and SentenceTransformer embedding encoder

In [8]:
# initialize ChromaDB in memory and create collection
import chromadb
from sentence_transformers import SentenceTransformer
import torch


client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(
    name="placement_questions",
    metadata={"hnsw:space": "cosine"},
)

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("Using device:", device)

# update collection with chunked data
texts = [record["text"] for record in chunked_data]
embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True).tolist()
added_ids = []

for idx, (record, embedding) in enumerate(zip(chunked_data, embeddings)):
    chunk_id = f"{record.get('filename', 'document')}_chunk_{record['chunk_index']}_{idx}"
    metadata = {k: v for k, v in record.items() if k != "text"}
    collection.add(
        documents=[record["text"]],
        metadatas=[metadata],
        ids=[chunk_id],
        embeddings=[embedding],
    )
    added_ids.append(chunk_id)

print("Total documents in collection:", collection.count())
if added_ids:
    print("Sample document from collection:", collection.get(ids=[added_ids[0]]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda


Batches:   0%|          | 0/319 [00:00<?, ?it/s]

Total documents in collection: 10186
Sample document from collection: {'ids': ['PTQB 2025-26_AInDS.csv_chunk_0_0'], 'embeddings': None, 'documents': ['            company name                                                                                                                                                                                '], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'chunk_index': 0, 'text_length': 76811, 'filename': 'PTQB 2025-26_AInDS.csv'}]}


In [10]:
# print a sample encoding
print("sample encoding:", embeddings[0])

sample encoding: [-0.09141591936349869, 0.00969727523624897, -0.0670882910490036, -0.0002484219439793378, -0.03863079845905304, -0.05445023253560066, 0.12293731421232224, -0.012057467363774776, 0.00015362507838290185, -0.11477953940629959, 0.0219353549182415, -0.0400148406624794, -0.025219179689884186, -0.041421182453632355, -0.08254433423280716, -0.04637456685304642, 0.06240537390112877, 0.029961194843053818, -0.010625522583723068, -0.11733835935592651, -0.032666176557540894, -0.008915639482438564, -0.015031998977065086, 0.007396976463496685, 0.008932136930525303, 0.041617073118686676, -0.021751267835497856, 0.06791029870510101, -0.024739401414990425, -0.14539377391338348, 0.009436596184968948, -0.009326882660388947, 0.11098844558000565, -0.001036031753756106, 0.0176333487033844, 0.009473573416471481, -0.13480088114738464, -0.006855376064777374, -0.002919156802818179, -0.01857095956802368, -0.010161135345697403, -0.02746206894516945, -0.03329164907336235, -0.04699746519327164, 0.01081

In [ ]:
# Add embeddings to the Vector Database
chromadb.UpdateCollectionMetadata
